# B-001 bis B-006: Fabric IQ und Basic direkt in Foundry vergleichen


Alle Zellen von oben nach unten ausfuehren. Die Referenz wird unter
`project-planning-business-reference-cases` veroeffentlicht und per ID gepinnt.
Foundry liest dieses Dataset, fragt je Agent alle sechs Fragen ab und bewertet
die neu erzeugten Antworten mit dem Custom-Evaluator v11.

Nur `query` erreicht die Agents. Referenzantworten, Kontext und Kriterien erhaelt
ausschliesslich der Judge. Azure-CLI-Anmeldung, Foundry-Zugriff und aktive
Fabric-Kapazitaet werden vorausgesetzt. Modellaufrufe verursachen Kosten.

Es gibt keine vorgelagerte Capture-Phase und keine Inline-Eingaben.
Jede Agent-Zelle startet einen neuen Foundry-Lauf mit sechs Antworten und Urteilen.
Bei Timeout den gespeicherten Run mit `--resume` fortsetzen, statt neu zu starten.
Die Berichte werden unter festen Dateinamen gespeichert; die letzte Zelle
vergleicht ausschliesslich die Foundry-Urteile.

Die Einzeltest-Notebooks bleiben separate Antwort- und Trace-Diagnosen.
Ein Gesamtdurchlauf beweist keine allgemeine Ueberlegenheit eines Agenten.


In [1]:
ROOT=$(git rev-parse --show-toplevel) &&
cd "$ROOT" &&
PYTHON="$ROOT/.venv/bin/python" &&
RUNNER="$ROOT/scripts/evaluate_business.py" &&
REFERENCE_SOURCE="$ROOT/evaluations/business_tests.json" &&
PROJECT_ENDPOINT="https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning" &&
IQ_AGENT=project-planning-agent && IQ_VERSION=8 &&
BASIC_AGENT=project-planning-baseline && BASIC_VERSION=6 &&
JUDGE_MODEL=gpt-4.1 && EVALUATOR_VERSION=13 &&
RUN_DIR="$ROOT/evaluations/notebook_runs/all-business-tests" &&
REFERENCE_ASSET_FILE="$RUN_DIR/reference.asset.json" &&
DATASET="$RUN_DIR/reference.asset.data.jsonl" &&
COMPARISON_ID=$(date -u +%Y%m%dT%H%M%SZ) &&
IQ_RESULT_FILE="$RUN_DIR/fabric-iq.evaluation.json" &&
BASIC_RESULT_FILE="$RUN_DIR/basic.evaluation.json" &&
SUMMARY_FILE="$RUN_DIR/comparison.json" &&
mkdir -p "$RUN_DIR" &&
printf 'Vergleichs-ID: %s\n' "$COMPARISON_ID"


Vergleichs-ID: 20261117T111017Z


Dateien: [Testdaten](../evaluations/business_tests.json),
[Runner](../scripts/evaluate_business.py).
Ausgaben: [Suite-Ordner](../evaluations/notebook_runs/all-business-tests/).


## Versionierte Foundry-Referenz

Die lokale Definition wird unter ihrem Inhaltshash registriert. Anschliessend wird die feste Foundry-Version heruntergeladen und geprueft. Bestehende Versionen werden wiederverwendet.

In [2]:
"$PYTHON" "$RUNNER" --endpoint "$PROJECT_ENDPOINT" \
  --publish-reference --dataset "$REFERENCE_SOURCE" \
  --result-file "$REFERENCE_ASSET_FILE" --show-endpoints &&
jq -M '{name,version,id}' "$REFERENCE_ASSET_FILE" &&
jq -s '[.[] | {id,title}]' "$DATASET"


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879?api-version=v1


Foundry dataset: project-planning-business-reference-cases version 87ca3d6024b4f879


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879?api-version=v1


HTTP POST https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879/credentials?api-version=v1


{


  "id": "azureai://accounts/ai-projplan-axowf3/projects/project-planning/data/project-planning-business-reference-cases/versions/87ca3d6024b4f879",


... (31 outputs left out)

    "id": "B-006",


    "title": "Project lookup through the qwert ontology synonym"


  }


]


Dateien: [Quelle](../evaluations/business_tests.json), [Asset-ID und Version](../evaluations/notebook_runs/all-business-tests/reference.asset.json), [Gepruefte Foundry-Kopie](../evaluations/notebook_runs/all-business-tests/reference.asset.data.jsonl), [Runner](../scripts/evaluate_business.py), [Dataset-Verwaltung](../scripts/business_evaluation_datasets.py).

## Fabric IQ: Agent aufrufen und bewerten


In [3]:
"$PYTHON" "$RUNNER" --endpoint "$PROJECT_ENDPOINT" \
  --agent "$IQ_AGENT" --agent-version "$IQ_VERSION" \
  --reference-asset "$REFERENCE_ASSET_FILE" --judge-model "$JUDGE_MODEL" \
  --evaluator-version "$EVALUATOR_VERSION" --result-file "$IQ_RESULT_FILE" \
  --run-label "Fabric IQ" --comparison-id "$COMPARISON_ID" \
  --show-endpoints


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879?api-version=v1


HTTP POST https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879/credentials?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/agents/project-planning-agent/versions/8?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/evaluators/project-planning-business-correctness/versions/13?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/evaluators/project-planning-business-correctness/versions/13?api-version=v1


Foundry evaluator: project-planning-business-correctness version 13


... (89 outputs left out)

    }


  },


  "note": "Counts summarize Foundry business judgments, not local grading. A single diagnostic run does not establish general superiority or isolate causality."


}


Eingaben: [Referenz](../evaluations/business_tests.json),
[Evaluator](../evaluations/evaluators/business_correctness.json),
[Runner](../scripts/evaluate_business.py).
Ausgabe: [Fabric IQ-Bericht](../evaluations/notebook_runs/all-business-tests/fabric-iq.evaluation.json)
mit sechs generierten Antworten unter `output_items[].sample` und sechs Foundry-Urteilen.

Bei Timeout: `evaluate_business.py --endpoint ... --resume <Ergebnisdatei>`.
Ein erneuter Start dieser Zelle erzeugt neue Antworten, keine Wiederbewertung alter Captures.


## Basic: Agent aufrufen und bewerten


In [4]:
"$PYTHON" "$RUNNER" --endpoint "$PROJECT_ENDPOINT" \
  --agent "$BASIC_AGENT" --agent-version "$BASIC_VERSION" \
  --reference-asset "$REFERENCE_ASSET_FILE" --judge-model "$JUDGE_MODEL" \
  --evaluator-version "$EVALUATOR_VERSION" --result-file "$BASIC_RESULT_FILE" \
  --run-label "Basic" --comparison-id "$COMPARISON_ID" \
  --show-endpoints


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879?api-version=v1


HTTP POST https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/datasets/project-planning-business-reference-cases/versions/87ca3d6024b4f879/credentials?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/agents/project-planning-baseline/versions/6?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/evaluators/project-planning-business-correctness/versions/13?api-version=v1


HTTP GET https://ai-projplan-axowf3.services.ai.azure.com/api/projects/project-planning/evaluators/project-planning-business-correctness/versions/13?api-version=v1


Foundry evaluator: project-planning-business-correctness version 13


... (72 outputs left out)

    }


  },


  "note": "Counts summarize Foundry business judgments, not local grading. A single diagnostic run does not establish general superiority or isolate causality."


}


Eingaben: [Referenz](../evaluations/business_tests.json),
[Evaluator](../evaluations/evaluators/business_correctness.json),
[Runner](../scripts/evaluate_business.py).
Ausgabe: [Basic-Bericht](../evaluations/notebook_runs/all-business-tests/basic.evaluation.json)
mit sechs generierten Antworten unter `output_items[].sample` und sechs Foundry-Urteilen.

Bei Timeout: `evaluate_business.py --endpoint ... --resume <Ergebnisdatei>`.
Ein erneuter Start dieser Zelle erzeugt neue Antworten, keine Wiederbewertung alter Captures.


## Vergleich

Nur Foundry-Urteile anzeigen, keine lokale Bewertung. Ein Gesamtdurchlauf
beweist keine allgemeine Ueberlegenheit; die Agents haben unterschiedliche
Zugriffswege und nur Fabric IQ kennt die zusaetzlichen Ontologie-Regeln.


In [5]:
COMPARISON=$(jq -e -s '
  .[0] as $iq | .[1] as $basic |
  ((($iq.comparison_id | type) == "string" and
    $iq.comparison_id == $basic.comparison_id and
    $iq.dataset.reference_sha256 == $basic.dataset.reference_sha256 and
    $iq.evaluator == $basic.evaluator and
    ([$iq.dataset.cases[].case_id] | sort) == ([$basic.dataset.cases[].case_id] | sort))
    or error("Reports must use the same comparison ID, reference, cases and evaluator")) |
  [$iq,$basic] | [.[] | .comparison.agents | to_entries[] | .key as $agent |
    .value.cases | to_entries[] |
    {agent:$agent,case_id:.key,passed:.value.passed,anchor:.value.anchor_score,reason:.value.reason}]' \
  "$IQ_RESULT_FILE" "$BASIC_RESULT_FILE") &&
printf '%s\n' "$COMPARISON" > "$SUMMARY_FILE" &&
printf 'Judge-Urteil und Anker-Anteil (Anteil gefundener Pflicht-IDs/-Zahlen)\n' &&
printf '%-16s%-16s%-16s\n' Test "Fabric IQ v$IQ_VERSION" "Basic v$BASIC_VERSION" &&
jq -r --arg iq "$IQ_AGENT:v$IQ_VERSION" --arg basic "$BASIC_AGENT:v$BASIC_VERSION" '
  group_by(.case_id)[] | . as $rows |
  [.[0].case_id,
   ($rows[] | select(.agent == $iq) | ({"true":"PASS","false":"FAIL"}[.passed | tostring]) + (if .anchor == null then "" else " " + ((.anchor * 100 | round) / 100 | tostring) end)),
   ($rows[] | select(.agent == $basic) | ({"true":"PASS","false":"FAIL"}[.passed | tostring]) + (if .anchor == null then "" else " " + ((.anchor * 100 | round) / 100 | tostring) end))]
  | map(. + (" " * ([1, 16 - length] | max))) | join("")' <<< "$COMPARISON" &&
jq -r '.comparison.agents | to_entries[] |
  "\(.key): \(.value.passed)/\(.value.passed + .value.failed) bestanden"' \
  "$IQ_RESULT_FILE" "$BASIC_RESULT_FILE" &&
jq -r '"Fabric IQ Report: \(.summary.report_url)"' "$IQ_RESULT_FILE" &&
jq -r '"Basic Report: \(.summary.report_url)"' "$BASIC_RESULT_FILE"


Judge-Urteil und Anker-Anteil (Anteil gefundener Pflicht-IDs/-Zahlen)


Test            Fabric IQ v8    Basic v6        


B-001           PASS 1          FAIL 1          


B-002           PASS 1          FAIL 0.5        


B-003           PASS 1          PASS 1          


B-004           PASS 1          PASS 1          


... (2 outputs left out)

project-planning-agent:v8: 5/6 bestanden


project-planning-baseline:v6: 2/6 bestanden


Fabric IQ Report: https://ai.azure.com/nextgen/r/mFJSZB60ST-YPRajMMqn9g,rg-projplan-demo,,ai-projplan-axowf3,project-planning/build/evaluations/eval_567b617db2974332a248975efd9e45b3/run/evalrun_4a341e7c88b24afcbf10fb7c7868d0c5


Basic Report: https://ai.azure.com/nextgen/r/mFJSZB60ST-YPRajMMqn9g,rg-projplan-demo,,ai-projplan-axowf3,project-planning/build/evaluations/eval_bcd6c74397824a64bd01815c4ff1d8a4/run/evalrun_a20cbd9487d9476f94db4333125c28f3


Eingaben: [Fabric-IQ-Bewertung](../evaluations/notebook_runs/all-business-tests/fabric-iq.evaluation.json),
[Basic-Bewertung](../evaluations/notebook_runs/all-business-tests/basic.evaluation.json).
Ausgabe: [Vergleich](../evaluations/notebook_runs/all-business-tests/comparison.json).
